<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - `examples.py` simulation

## Path Constants

Define the project name.

In [1]:
PROJECT = "cl_st1_ph1_tatiana"

In [2]:
from pathlib import Path

SCORES_FILE = Path(f"sas/output_{PROJECT}/{PROJECT}_scores_only.tsv")
FILE_IDS_PATH = Path("file_ids.txt")

## Load factor scores

In [3]:
import pandas as pd

scores_df = pd.read_csv(SCORES_FILE, sep="\t")
scores_df = scores_df.rename(columns={"filename": "file_id"})

scores_df

,file_id,fac1,fac2,fac3,fac4,fac5,fac6
0,t000001,-1,0,7,0,1,0
1,t000002,4,0,0,0,1,0
2,t000003,0,2,0,0,0,0
3,t000004,0,14,0,0,0,0
4,t000005,0,0,0,0,0,1
...,...,...,...,...,...,...,...
14090,t014572,0,0,0,0,1,-3
14091,t014573,0,0,0,0,1,-3
14092,t014574,0,12,0,0,0,0
14093,t014575,0,0,0,0,1,-2


## Load file ID mapping

In [4]:
file_ids_df = pd.read_csv(
    FILE_IDS_PATH,
    sep=" ",
    names=["file_id", "group_filename"],
)

file_ids_df.head()

,file_id,group_filename
0,t000001,tweet_000002_000001.txt
1,t000002,tweet_000003_000001.txt
2,t000003,tweet_000004_000001.txt
3,t000004,tweet_000007_000001.txt
4,t000005,tweet_000010_000001.txt


## Merge factor scores with file ID mapping

In [5]:
scores_file_ids_df = scores_df.merge(file_ids_df, on="file_id", how="left")
scores_file_ids_df = scores_file_ids_df[
    ["file_id", "group_filename"]
    + [col for col in scores_file_ids_df.columns if col not in ["file_id", "group_filename"]]
    ]

scores_file_ids_df


,file_id,group_filename,fac1,fac2,fac3,fac4,fac5,fac6
0,t000001,tweet_000002_000001.txt,-1,0,7,0,1,0
1,t000002,tweet_000003_000001.txt,4,0,0,0,1,0
2,t000003,tweet_000004_000001.txt,0,2,0,0,0,0
3,t000004,tweet_000007_000001.txt,0,14,0,0,0,0
4,t000005,tweet_000010_000001.txt,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...
14090,t014572,tweet_026773_000002.txt,0,0,0,0,1,-3
14091,t014573,tweet_026773_000003.txt,0,0,0,0,1,-3
14092,t014574,tweet_026774_000001.txt,0,12,0,0,0,0
14093,t014575,tweet_026775_000001.txt,0,0,0,0,1,-2


## Simulate selection process

### List factors

In [6]:
import re

factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

print("\n".join(
    f"{'' if i == 0 else '#'}FACTOR = {factor!r}"
    for i, factor in enumerate(factor_cols)
))

FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'
#FACTOR = 'fac5'
#FACTOR = 'fac6'


### Copy/Paste the groups list and Uncomment/Comment groups accordingly

In `examples.py`, texts with a factor score of exactly `0` are skipped. In this simulation tool, they are not, for the sake of simplicity and to allow for a broader view.

In [7]:
FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'
#FACTOR = 'fac5'
#FACTOR = 'fac6'

POLE = "positive"
#POLE = "negative"

N_EXAMPLES = 40 # Define the number of examples to select

if POLE == "positive":
    examples_df = scores_file_ids_df.nlargest(N_EXAMPLES, FACTOR)
elif POLE == "negative":
    examples_df = scores_file_ids_df.nsmallest(N_EXAMPLES, FACTOR)
else:
    raise ValueError(f"Unknown POLE: {POLE!r}")

examples_df = examples_df[["file_id", "group_filename"] + factor_cols]

examples_df

,file_id,group_filename,fac1,fac2,fac3,fac4,fac5,fac6
10780,t011142,tweet_020511_000001.txt,30,0,0,0,0,1
12603,t013037,tweet_023927_000001.txt,30,0,0,0,0,0
6934,t007170,tweet_013148_000001.txt,28,0,0,1,0,0
8407,t008695,tweet_016003_000001.txt,28,0,0,1,0,0
11674,t012076,tweet_022182_000001.txt,27,0,0,0,0,0
6924,t007160,tweet_013127_000001.txt,26,0,0,0,0,1
1381,t001424,tweet_002576_000001.txt,25,0,0,0,0,0
12241,t012660,tweet_023266_000001.txt,25,0,0,0,0,0
6031,t006237,tweet_011479_000001.txt,23,0,0,0,0,0
5309,t005498,tweet_010120_000002.txt,22,0,1,0,0,0


## Fetch information from `examples/`

Write a Jupyter Notebook cell that parses generated LaTeX example files under `examples/` and builds a pandas DataFrame.

Requirements:

- Search only immediate subdirectories of `examples/` whose names match `f<n>_<pos|neg>`.
- Sort subdirectories by numeric factor number, then by pole order: `pos`, `neg`.
- In each matching subdirectory, parse only `.tex` files whose names match `f<n>_<pos|neg>_<m>.tex`, where the factor and pole match the parent directory.
- Sort files by numeric sequence `<m>`.
- For each file, read the first line that starts with `\begin{textsample}{`.
- Convert:
  - `POS` to `pole = "positive"`
  - `NEG` to `pole = "negative"`
  - `Dim <n>` to `factor = "fac<n>"`
  - escaped LaTeX underscores, such as `\_`, to `_` in `group_filename`
  - `Score <n>` to numeric `score`

- Create a pandas DataFrame with exactly these columns:

      example_filename
      factor
      pole
      group_filename
      score

- The DataFrame should be sorted in the order of subdirectory/file parsing.
- Raise clear errors for matching files that lack a `\begin{textsample}{...}` line.
- After creating the DataFrame, export it to:
  - `examples/examples_summary.ndjson` as newline-delimited JSON, preserving non-ASCII characters.
  - `examples/examples_summary.xlsx` as an Excel file without the DataFrame index.
- Print a confirmation message for each exported file, including the number of records written.

In [8]:
import re
from pathlib import Path

import pandas as pd

EXAMPLES_DIR = Path("examples")

dir_pattern = re.compile(r"^f(?P<factor_num>\d+)_(?P<pole_code>pos|neg)$")
file_pattern_template = r"^f{factor_num}_{pole_code}_(?P<seq>\d+)\.tex$"

textsample_prefix = r"\begin{textsample}{"
textsample_pattern = re.compile(
    r"""^\\begin\{textsample\}\{
        (?P<pole_label>POS|NEG)
        \s+Dim\s+
        (?P<dim>\d+)
        \s+–\s+
        Score\s+
        (?P<score>[+-]?(?:\d+(?:\.\d*)?|\.\d+))
        \s+–\s+
        (?P<group_filename>.+?)
        \}
    """,
    re.VERBOSE,
)

pole_name = {
    "POS": "positive",
    "NEG": "negative",
}

pole_order = {
    "pos": 0,
    "neg": 1,
}

rows = []

if not EXAMPLES_DIR.exists():
    raise FileNotFoundError(f"Examples directory not found: {EXAMPLES_DIR}")

example_dirs = []

for path in EXAMPLES_DIR.iterdir():
    if not path.is_dir():
        continue

    dir_match = dir_pattern.fullmatch(path.name)
    if not dir_match:
        continue

    example_dirs.append(
        {
            "path": path,
            "factor_num": int(dir_match.group("factor_num")),
            "pole_code": dir_match.group("pole_code"),
        }
    )

example_dirs = sorted(
    example_dirs,
    key=lambda item: (item["factor_num"], pole_order[item["pole_code"]]),
)

for example_dir in example_dirs:
    factor_num = example_dir["factor_num"]
    pole_code = example_dir["pole_code"]
    dir_path = example_dir["path"]

    file_pattern = re.compile(
        file_pattern_template.format(
            factor_num=factor_num,
            pole_code=pole_code,
        )
    )

    tex_files = []

    for tex_path in dir_path.iterdir():
        if not tex_path.is_file():
            continue

        file_match = file_pattern.fullmatch(tex_path.name)
        if not file_match:
            continue

        tex_files.append(
            {
                "path": tex_path,
                "seq": int(file_match.group("seq")),
            }
        )

    tex_files = sorted(tex_files, key=lambda item: item["seq"])

    for tex_file in tex_files:
        tex_path = tex_file["path"]

        textsample_line = None

        with open(tex_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith(textsample_prefix):
                    textsample_line = line
                    break

        if textsample_line is None:
            raise ValueError(
                f"No line beginning with {textsample_prefix!r} found in {tex_path}"
            )

        textsample_match = textsample_pattern.match(textsample_line)

        if not textsample_match:
            raise ValueError(
                "Could not parse textsample line in "
                f"{tex_path}:\n{textsample_line}"
            )

        pole_label = textsample_match.group("pole_label")
        dim = int(textsample_match.group("dim"))

        if dim != factor_num:
            raise ValueError(
                f"Factor mismatch in {tex_path}: "
                f"directory says f{factor_num}, textsample says Dim {dim}"
            )

        expected_pole_label = pole_code.upper()
        if pole_label != expected_pole_label:
            raise ValueError(
                f"Pole mismatch in {tex_path}: "
                f"directory says {pole_code}, textsample says {pole_label}"
            )

        group_filename = textsample_match.group("group_filename").replace(r"\_", "_").strip()
        score = float(textsample_match.group("score"))

        rows.append(
            {
                "example_filename": tex_path.name,
                "factor": f"fac{dim}",
                "pole": pole_name[pole_label],
                "group_filename": group_filename,
                "score": score,
            }
        )

examples_summary_df = pd.DataFrame(
    rows,
    columns=[
        "example_filename",
        "factor",
        "pole",
        "group_filename",
        "score",
    ],
)

# Export to NDJSON and Excel
from pathlib import Path

NDJSON_OUT = Path("examples/examples_summary.ndjson")
EXCEL_OUT = Path("examples/examples_summary.xlsx")

examples_summary_df.to_json(
    NDJSON_OUT,
    orient="records",
    lines=True,
    force_ascii=False,
)

examples_summary_df.to_excel(
    EXCEL_OUT,
    index=False,
)

print(f"Wrote {len(examples_summary_df):,} records to {NDJSON_OUT}")
print(f"Wrote {len(examples_summary_df):,} records to {EXCEL_OUT}")

examples_summary_df

Wrote 480 records to examples/examples_summary.ndjson
Wrote 480 records to examples/examples_summary.xlsx


,example_filename,factor,pole,group_filename,score
0,f1_pos_001.tex,fac1,positive,t013037,30.0
1,f1_pos_002.tex,fac1,positive,t011142,30.0
2,f1_pos_003.tex,fac1,positive,t007170,28.0
3,f1_pos_004.tex,fac1,positive,t008695,28.0
4,f1_pos_005.tex,fac1,positive,t012076,27.0
...,...,...,...,...,...
475,f6_neg_036.tex,fac6,negative,t009720,-8.0
476,f6_neg_037.tex,fac6,negative,t012913,-8.0
477,f6_neg_038.tex,fac6,negative,t008423,-8.0
478,f6_neg_039.tex,fac6,negative,t009948,-8.0
